# create one stop borders gpkg

In [ ]:
from pathlib import Path
import os
import re
import sqlite3
import unicodedata
import difflib
import urllib.request

import geopandas as gpd
import pandas as pd

# Input richiesto
gpkg_path = Path.home() / "Desktop" / "osbp_points.gpkg"
layer_name = "osbp_points"

if not gpkg_path.exists():
    raise FileNotFoundError(f"File non trovato: {gpkg_path}")


def normalize_country_name(value: str) -> str:
    text = unicodedata.normalize("NFKD", str(value))
    text = "".join(ch for ch in text if not unicodedata.combining(ch))
    text = text.lower().strip()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text


def build_country_iso3_lookup() -> dict:
    lookup = {}

    # Natural Earth names -> ISO3
    cache_dir = Path.home() / ".geopandas_cache"
    cache_dir.mkdir(exist_ok=True)
    ne_zip = cache_dir / "ne_10m_admin_0_countries.zip"
    if not ne_zip.exists():
        ne_url = "https://naciscdn.org/naturalearth/10m/cultural/ne_10m_admin_0_countries.zip"
        urllib.request.urlretrieve(ne_url, ne_zip)

    world = gpd.read_file(ne_zip)

    iso_candidates = [c for c in ["ISO_A3", "ADM0_A3", "SOV_A3", "GU_A3"] if c in world.columns]
    if not iso_candidates:
        raise KeyError("Nessuna colonna ISO3 trovata nel dataset Natural Earth")
    iso_col = iso_candidates[0]

    name_candidates = [
        c for c in ["NAME", "NAME_LONG", "NAME_EN", "ADMIN", "SOVEREIGNT", "BRK_NAME", "FORMAL_EN"] if c in world.columns
    ]

    for _, row in world.iterrows():
        iso = str(row[iso_col]).strip().upper()
        if len(iso) != 3:
            continue
        for c in name_candidates:
            val = row[c]
            if pd.isna(val):
                continue
            lookup[normalize_country_name(val)] = iso

    # Alias frequenti
    aliases = {
        "ivory coast": "CIV",
        "cote d ivoire": "CIV",
        "cote divoire": "CIV",
        "democratic republic of congo": "COD",
        "dr congo": "COD",
        "drc": "COD",
        "republic of congo": "COG",
        "congo republic": "COG",
        "cape verde": "CPV",
        "cabo verde": "CPV",
        "eswatini": "SWZ",
        "swaziland": "SWZ",
        "south sudan": "SSD",
        "sao tome and principe": "STP",
        "equatorial guinea": "GNQ",
        "gambia": "GMB",
        "the gambia": "GMB",
    }
    for k, v in aliases.items():
        lookup[normalize_country_name(k)] = v

    return lookup


lookup = build_country_iso3_lookup()
lookup_keys = list(lookup.keys())


def to_iso3(value):
    if pd.isna(value):
        return None
    text = str(value).strip()
    if len(text) == 3 and text.isalpha():
        return text.upper()

    key = normalize_country_name(text)
    if key in lookup:
        return lookup[key]

    matches = difflib.get_close_matches(key, lookup_keys, n=1, cutoff=0.90)
    if matches:
        return lookup[matches[0]]
    return None


# Leggi layer
gdf = gpd.read_file(gpkg_path, layer=layer_name)
missing_cols = [c for c in ["country1", "country2"] if c not in gdf.columns]
if missing_cols:
    raise KeyError(f"Colonne mancanti nel layer {layer_name}: {missing_cols}")

# Converti country1/country2 in ISO3
gdf["country1"] = gdf["country1"].apply(to_iso3)
gdf["country2"] = gdf["country2"].apply(to_iso3)

unmatched_c1 = sorted(gdf.loc[gdf["country1"].isna(), "country1"].dropna().astype(str).unique())
unmatched_c2 = sorted(gdf.loc[gdf["country2"].isna(), "country2"].dropna().astype(str).unique())

# Scrivi su temp e sostituisci in modo sicuro
tmp_path = gpkg_path.with_name(gpkg_path.stem + "_tmp.gpkg")
if tmp_path.exists():
    tmp_path.unlink()

gdf.to_file(tmp_path, layer=layer_name, driver="GPKG")

applied_path = gpkg_path
try:
    os.replace(tmp_path, gpkg_path)
except PermissionError:
    applied_path = tmp_path
    print("[warn] osbp_points.gpkg bloccato da un altro processo.")
    print(f"[warn] File aggiornato salvato come: {tmp_path}")

print(f"[save] File aggiornato: {applied_path}")
print(f"[info] country1 null dopo conversione: {int(gdf['country1'].isna().sum())}")
print(f"[info] country2 null dopo conversione: {int(gdf['country2'].isna().sum())}")
print(gdf[["country1", "country2"]].head())

# find coordinates from osbp border points names

In [1]:
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut
import time

def get_coordinates(border_name, country):
    geolocator = Nominatim(user_agent="osbp_geocoder_script")
    # Clean the name (e.g., "Namanga/Namanga" -> "Namanga")
    primary_name = border_name.split('/')[0].strip()
    query = f"{primary_name}, {country}"
    
    try:
        location = geolocator.geocode(query, timeout=10)
        if location:
            return location.latitude, location.longitude
        return None, None
    except GeocoderTimedOut:
        return None, None

# Load the file
file_path = "osbp_borders.xlsx"
# Note: Ensure you read the correct sheet. Since your upload was converted to a CSV locally,
# adjust the filename below to match your actual Excel file.
df = pd.read_excel(file_path)

# Add empty columns
df['Latitude'] = None
df['Longitude'] = None

# Iterate and fetch coordinates
print("Fetching coordinates...")
for index, row in df.iterrows():
    border = row['Border Crossing (OSBP Name)']
    country = row['Country 1 (Country A)']
    
    lat, lon = get_coordinates(border, country)
    df.at[index, 'Latitude'] = lat
    df.at[index, 'Longitude'] = lon
    
    # Nominatim requires a 1-second delay between requests
    time.sleep(1)

# Save the updated file
output_path = "osbp_borders_with_coordinates_dupl.xlsx"
df.to_excel(output_path, index=False)
print(f"Done. Saved to {output_path}")

Fetching coordinates...
Done. Saved to osbp_borders_with_coordinates.xlsx


# create geodatabase from excel

In [5]:
from pathlib import Path

import geopandas as gpd
import pandas as pd

# =========================
# Configurazione
# =========================
INPUT_XLSX = Path("osbp_borders_with_coordinates_dupl.xlsx")
if not INPUT_XLSX.exists():
    INPUT_XLSX = Path("osbp_borders_with_coordinates.xlsx")

OUTPUT_GPKG = INPUT_XLSX.with_suffix(".gpkg")
OUTPUT_LAYER = "osbp_points"


def _find_column(df: pd.DataFrame, candidates: list[str]):
    normalized = {str(c).strip().lower(): c for c in df.columns}
    for c in candidates:
        if c.lower() in normalized:
            return normalized[c.lower()]
    return None


df = pd.read_excel(INPUT_XLSX)
print(f"[info] Righe lette da Excel: {len(df)}")

lat_col = _find_column(df, ["Latitude", "lat"])
lon_col = _find_column(df, ["Longitude", "lon", "lng"])
if lat_col is None or lon_col is None:
    raise ValueError("Colonne Latitude/Longitude non trovate nel file Excel")

df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")
df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
valid = df[lat_col].notna() & df[lon_col].notna()
df_valid = df.loc[valid].copy()

if df_valid.empty:
    raise ValueError("Nessuna riga con coordinate valide")

# Rinomina Country 1 / Country 2 senza conversione valori
country1_src = _find_column(df_valid, ["Country 1", "Country 1 (Country A)", "country1", "country 1"])
country2_src = _find_column(df_valid, ["Country 2", "Country 2 (Country B)", "country2", "country 2"])

if country1_src is not None and country1_src != "country1":
    df_valid = df_valid.rename(columns={country1_src: "country1"})

if country2_src is not None and country2_src != "country2":
    df_valid = df_valid.rename(columns={country2_src: "country2"})

# GeoDataFrame con tutte le altre colonne mantenute
gdf = gpd.GeoDataFrame(
    df_valid,
    geometry=gpd.points_from_xy(df_valid[lon_col], df_valid[lat_col]),
    crs=4326,
)

if OUTPUT_GPKG.exists():
    OUTPUT_GPKG.unlink()

gdf.to_file(OUTPUT_GPKG, layer=OUTPUT_LAYER, driver="GPKG")

print(f"[save] GeoPackage creato: {OUTPUT_GPKG}")
print(f"[save] Layer: {OUTPUT_LAYER}")
print(f"[save] Punti esportati: {len(gdf)} su {len(df)}")
gdf.head()

[info] Righe lette da Excel: 84
[save] GeoPackage creato: osbp_borders_with_coordinates.gpkg
[save] Layer: osbp_points
[save] Punti esportati: 84 su 84


,Border Crossing (OSBP Name),country1,country2,Latitude,Longitude,geometry
0,Namanga/Namanga,Kenya,Tanzania,-2.546715,36.797438,POINT (36.79744 -2.54672)
1,Rusumo/Rusumo,Rwanda,Tanzania,-1.673419,30.092364,POINT (30.09236 -1.67342)
2,Malaba/Malaba,Kenya,Uganda,0.632706,34.269398,POINT (34.2694 0.63271)
3,Taveta/Holili,Kenya,Tanzania,-3.400000,37.683300,POINT (37.6833 -3.4)
4,Lunga Lunga / Horo Horo,Kenya,Tanzania,-4.353398,38.962264,POINT (38.96226 -4.3534)


# manual adjust error points

In [10]:
from pathlib import Path
import geopandas as gpd
import pandas as pd

# =========================
# Error flags da distanza al confine piu vicino
# =========================
INPUT_GPKG = Path("osbp_borders_with_coordinates_errors.gpkg")
INPUT_LAYER = "osbp_points"

# Opzione 2: confini Natural Earth scaricati al volo
BORDERS_URL = "https://naturalearth.s3.amazonaws.com/10m_cultural/ne_10m_admin_0_countries.zip"

# Salva in nuovo file per evitare lock del GPKG aperto in Windows
OUTPUT_GPKG = Path("osbp_borders_with_coordinates_errors.gpkg")
OUTPUT_LAYER = INPUT_LAYER


# 1) Carica punti
points = gpd.read_file(INPUT_GPKG, layer=INPUT_LAYER)
if points.crs is None:
    points = points.set_crs(4326, allow_override=True)
else:
    points = points.to_crs(4326)

# 2) Carica confini
try:
    borders = gpd.read_file(BORDERS_URL)
except Exception:
    raise RuntimeError(
        "Impossibile caricare i confini da URL. Usa un file locale dei confini."
    )

if borders.crs is None:
    borders = borders.set_crs(4326, allow_override=True)
else:
    borders = borders.to_crs(4326)

# Usa il contorno dei poligoni come linea di confine
border_lines = borders.geometry.boundary
border_lines_gdf = gpd.GeoDataFrame(geometry=border_lines, crs=4326)
border_lines_gdf = border_lines_gdf[border_lines_gdf.geometry.notna() & ~border_lines_gdf.geometry.is_empty].copy()

# 3) Distanza minima al confine in metri
points_m = points.to_crs(3857)
border_lines_m = border_lines_gdf.to_crs(3857)

min_dist_m = []
for geom in points_m.geometry:
    d = border_lines_m.distance(geom).min()
    min_dist_m.append(float(d) if pd.notna(d) else pd.NA)

points["distance_to_border_m"] = min_dist_m
points["error_3"] = points["distance_to_border_m"] > 2000
points["error_5"] = points["distance_to_border_m"] > 5000
points["error_10"] = points["distance_to_border_m"] > 10000

# 4) Salva
if OUTPUT_GPKG.exists():
    OUTPUT_GPKG.unlink()

points.to_file(OUTPUT_GPKG, layer=OUTPUT_LAYER, driver="GPKG")

print(f"[save] Creato {OUTPUT_GPKG} layer={OUTPUT_LAYER}")
print(f"[info] error_3=True: {int(points['error_3'].sum())}")
print(f"[info] error_5=True: {int(points['error_5'].sum())}")
print(f"[info] error_10=True: {int(points['error_10'].sum())}")
points[["distance_to_border_m", "error_3", "error_5", "error_10"]].head()

[save] Creato osbp_borders_with_coordinates_errors.gpkg layer=osbp_points
[info] error_3=True: 25
[info] error_5=True: 4
[info] error_10=True: 4


,distance_to_border_m,error_3,error_5,error_10
0,1967.766897,False,False,False
1,117.826236,False,False,False
2,3141.300121,True,False,False
3,1020.876080,False,False,False
4,1177.166016,False,False,False


# delete error columns

In [11]:
from pathlib import Path
import geopandas as gpd

input_gpkg = Path("osbp_borders_with_coordinates_errors.gpkg")
input_layer = "osbp_points"
output_gpkg = Path("osbp_points.gpkg")
output_layer = "osbp_points"

points = gpd.read_file(input_gpkg, layer=input_layer)

error_cols = [c for c in points.columns if str(c).startswith("error_")]
if error_cols:
    points = points.drop(columns=error_cols)

if output_gpkg.exists():
    output_gpkg.unlink()

points.to_file(output_gpkg, layer=output_layer, driver="GPKG")

print(f"[save] Output: {output_gpkg} (layer={output_layer})")
print(f"[info] Colonne error_* rimosse: {error_cols if error_cols else 'nessuna'}")
print(f"[info] Righe salvate: {len(points)}")
points.head()

[save] Output: osbp_points.gpkg (layer=osbp_points)
[info] Colonne error_* rimosse: ['error_3', 'error_5', 'error_10']
[info] Righe salvate: 82


,Border Crossing (OSBP Name),country1,country2,Latitude,Longitude,distance_to_border_m,geometry
0,Namanga/Namanga,Kenya,Tanzania,-2.546715,36.797438,1967.766897,POINT (36.80477 -2.55673)
1,Rusumo/Rusumo,Rwanda,Tanzania,-1.673419,30.092364,117.826236,POINT (30.78305 -2.37146)
2,Malaba/Malaba,Kenya,Uganda,0.632706,34.269398,3141.300121,POINT (34.2694 0.63271)
3,Taveta/Holili,Kenya,Tanzania,-3.400000,37.683300,1020.876080,POINT (37.63106 -3.3789)
4,Lunga Lunga / Horo Horo,Kenya,Tanzania,-4.353398,38.962264,1177.166016,POINT (39.11588 -4.6106)
